In [ ]:
import os
import time
import requests
import pandas as pd
from tqdm import tqdm
from bibtexparser.bwriter import BibTexWriter
from bibtexparser.bibdatabase import BibDatabase
import bibtexparser

# --- 1. CONFIGURATION ---
INPUT_CSV_FILE = 'beetle_citations.csv'
OUTPUT_DIR_PDFS = 'Downloaded_Open_Access_PDFs'
OUTPUT_BIB_DOWNLOADED = 'open_access_citations.bib'
OUTPUT_BIB_PAYWALLED = 'paywalled_citations.bib'
YOUR_EMAIL = "gmarais@ufl.edu"

# --- 2. HELPER FUNCTIONS (No changes) ---

def get_oa_pdf_url(doi, email):
    """Queries the Unpaywall API to find a legal Open Access PDF URL for a given DOI."""
    try:
        url = f"https://api.unpaywall.org/v2/{doi}?email={email}"
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        if data.get('best_oa_location') and data['best_oa_location'].get('url_for_pdf'):
            return data['best_oa_location']['url_for_pdf']
    except requests.exceptions.RequestException:
        pass
    return None

def sanitize_filename(name):
    """Removes illegal characters from a string to make it a valid filename."""
    if not isinstance(name, str):
        name = "No Title"
    illegal_chars = ['<', '>', ':', '"', '/', '\\', '|', '?', '*', '\n', '\r']
    for char in illegal_chars:
        name = name.replace(char, '_')
    return name[:150]

def download_pdf(pdf_url, filepath):
    """Downloads a PDF from a URL and saves it to a given path."""
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(pdf_url, headers=headers, stream=True, timeout=30)
        response.raise_for_status()
        with open(filepath, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        return True
    except requests.exceptions.RequestException:
        return False

def csv_row_to_bibtex_entry(row):
    """Converts a pandas DataFrame row (as a dictionary) to a bibtexparser entry format."""
    authors_str = ""
    if pd.notna(row.get('authors')):
        authors_list = [name.strip() for name in str(row['authors']).split(';')]
        authors_str = " and ".join(authors_list)

    year_str = str(int(row['year'])) if pd.notna(row.get('year')) else "ND"
    first_author_lastname = authors_str.split(' ')[0].split(',')[-1] if authors_str else "Unknown"
    
    # Clean the DOI for the citation key
    clean_doi = str(row.get('doi', '')).split('doi.org/')[-1].replace('/', '-')
    citation_key = f"{first_author_lastname}{year_str}_{clean_doi[:10]}"

    entry = {
        'ENTRYTYPE': 'article', 'ID': citation_key, 'author': authors_str,
        'title': str(row.get('title', 'No Title')), 'year': year_str,
        'journal': str(row.get('journal', '')), 'doi': str(row.get('doi', ''))
    }
    return entry

# --- 3. MAIN SCRIPT ---

if __name__ == "__main__":
    print("Starting Open Access PDF downloader script...")
    os.makedirs(OUTPUT_DIR_PDFS, exist_ok=True)
    
    # --- RESUMABILITY: LOAD PREVIOUS RESULTS ---
    processed_dois = set()
    
    def load_previous_db(filename):
        if os.path.exists(filename):
            try:
                with open(filename, 'r', encoding='utf-8') as bibfile:
                    db = bibtexparser.load(bibfile)
                    for entry in db.entries:
                        if 'doi' in entry:
                            processed_dois.add(entry['doi'].split('doi.org/')[-1])
                    print(f"Loaded {len(db.entries)} entries from '{filename}'.")
                    return db
            except Exception as e:
                print(f"Warning: Could not load or parse '{filename}'. Starting fresh. Error: {e}")
        return BibDatabase()

    downloaded_db = load_previous_db(OUTPUT_BIB_DOWNLOADED)
    paywalled_db = load_previous_db(OUTPUT_BIB_PAYWALLED)
    
    print(f"Found {len(processed_dois)} DOIs already processed in previous sessions. They will be skipped.")
    # --- END RESUMABILITY SETUP ---

    try:
        print(f"Loading data from '{INPUT_CSV_FILE}'...")
        df = pd.read_csv(INPUT_CSV_FILE)
        df.dropna(subset=['doi'], inplace=True)
        print("...Data loaded successfully!")
    except FileNotFoundError:
        print(f"ERROR: Input file '{INPUT_CSV_FILE}' not found.")
        exit()

    pbar = tqdm(df.iterrows(), total=len(df), desc="Finding and Downloading PDFs")
    
    for index, row in pbar:
        doi = str(row['doi']).strip()
        
        # Clean the DOI: remove the URL part if it exists for the lookup
        clean_doi = doi.split('doi.org/')[-1]
        
        # --- RESUMABILITY: SKIP IF ALREADY PROCESSED ---
        if clean_doi in processed_dois:
            continue
        
        pdf_url = get_oa_pdf_url(clean_doi, YOUR_EMAIL)
        bibtex_entry = csv_row_to_bibtex_entry(row)

        if pdf_url:
            title = row.get('title', 'No Title')
            filename = sanitize_filename(title) + '.pdf'
            filepath = os.path.join(OUTPUT_DIR_PDFS, filename)
            
            if download_pdf(pdf_url, filepath):
                downloaded_db.entries.append(bibtex_entry)
            else:
                paywalled_db.entries.append(bibtex_entry)
        else:
            paywalled_db.entries.append(bibtex_entry)
            
        # Add the newly processed DOI to our set so we don't re-process it in this session
        processed_dois.add(clean_doi)
        
        time.sleep(0.1)

    print("\n--- Processing Complete! ---")

    writer = BibTexWriter()
    writer.indent = '    '
    
    with open(OUTPUT_BIB_DOWNLOADED, 'w', encoding='utf-8') as bibfile:
        bibfile.write(writer.write(downloaded_db))
    print(f"✅ Total downloaded papers: {len(downloaded_db.entries)}.")
    print(f"   - PDFs are in '{OUTPUT_DIR_PDFS}'.")
    print(f"   - BibTeX list saved to '{OUTPUT_BIB_DOWNLOADED}'")

    with open(OUTPUT_BIB_PAYWALLED, 'w', encoding='utf-8') as bibfile:
        bibfile.write(writer.write(paywalled_db))
    print(f"\n🚫 Total paywalled/unfound papers: {len(paywalled_db.entries)}.")
    print(f"   - BibTeX list saved to '{OUTPUT_BIB_PAYWALLED}'")

Starting Open Access PDF downloader script...
Found 0 DOIs already processed in previous sessions. They will be skipped.
Loading data from 'beetle_citations.csv'...
...Data loaded successfully!


Finding and Downloading PDFs:   3%|▎         | 5030/158494 [55:56<28:26:41,  1.50it/s]  


OSError: [Errno 22] Invalid argument: 'Downloaded_Open_Access_PDFs\\Range extension of the Munchique rufous lancehead\r\nBothrocophias colombianus in Colombia.pdf'